# PyTorch: Using MLflow and Optuna for experiment tracking and hyperparameter optimization

In [114]:
import optuna
import mlflow
import torch

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torchvision import datasets, transforms
from torch.optim.lr_scheduler import StepLR
from mlflow import pytorch
from pprint import pformat

In [115]:
import matplotlib
matplotlib.use("Agg")      
import matplotlib.pyplot as plt
import numpy as np
import json, tempfile, os, time

In [116]:
# --- Point at a tracking server (if you have one running) ---
mlflow.set_tracking_uri(uri="http://127.0.0.1:5002")

# --- Create / set the experiment so all Optuna trials live together ---
EXPERIMENT_NAME = "mlflow-optuna-2"
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"MLflow tracking URI : {mlflow.get_tracking_uri()}")
print(f"Active experiment   : {EXPERIMENT_NAME}")

MLflow tracking URI : http://127.0.0.1:5002
Active experiment   : mlflow-optuna-2


In [117]:
class Net(nn.Module):
    def __init__(self, dropout=0.0):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.dropout1 = nn.Dropout2d(dropout)
        self.dropout2 = nn.Dropout2d(dropout)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.conv1(x)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = F.max_pool2d(x, 2)
        x = self.dropout1(x)
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.dropout2(x)
        x = self.fc2(x)
        output = F.log_softmax(x, dim=1)
        return output
    

In [118]:
# Training loop 
def train(model, device, train_loader, optimizer, epoch):
    model.train()
    train_set_size = len(train_loader.dataset)
    num_batches = len(train_loader)
    train_loss = 0.0
    correct = 0 
    start_time = time.time()
    forward_time, backward_time = 0.0, 0.0                                                   
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        train_loss += loss.item()
        pred = output.argmax(dim=1, keepdim=True)                 
        correct += pred.eq(target.view_as(pred)).sum().item()   
        forward_time += time.time() - start_time
        start_time = time.time()  
        loss.backward()
        backward_time += time.time() - start_time
        optimizer.step()
        start_time = time.time()
        if batch_idx % 10 == 0:
            batch_size = len(data)
            print(f"Train Epoch: {epoch} [{batch_idx * batch_size}/{train_set_size} "
                  f"({100. * batch_idx / num_batches:.0f}%)]\tLoss: {loss.item():.6f}")
    avg_train_loss = train_loss / num_batches
    avg_forward_time = forward_time / num_batches
    avg_backward_time = backward_time / num_batches
    train_acc = 100. * correct / train_set_size                      
    return avg_train_loss, train_acc, avg_forward_time, avg_backward_time

# Validation loop
def validate(model, device, val_loader):
    model.eval()
    val_set_size = len(val_loader.dataset)
    val_loss = 0
    correct = 0
    start_time = time.time()
    forward_time = 0.0
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            val_loss += F.nll_loss(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
            forward_time += time.time() - start_time
    val_loss /= val_set_size
    val_acc = 100. * correct / val_set_size  
    avg_val_forward_time = forward_time / len(val_loader)                       

    print(f"Test set: Average loss: {val_loss:.4f}, Accuracy: {correct}/{val_set_size} "
          f"({val_acc:.0f}%)\n")
    return val_loss, val_acc , avg_val_forward_time                                       

In [119]:
def log_training_curves(history: dict, trial_number: int):
    """Save a 2-panel training curves plot and log it as an MLflow artifact."""
    epochs = range(len(history["train_loss"]))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # --- Loss ---
    ax1.plot(epochs, history["train_loss"], label="Train Loss")
    ax1.plot(epochs, history["val_loss"],   label="Val Loss")
    ax1.set_xlabel("Epoch"); ax1.set_ylabel("Loss")
    ax1.set_title(f"Trial {trial_number} – Loss"); ax1.legend()

    # --- Accuracy ---
    ax2.plot(epochs, history["train_acc"], label="Train Acc")
    ax2.plot(epochs, history["val_acc"],   label="Val Acc")
    ax2.set_xlabel("Epoch"); ax2.set_ylabel("Accuracy (%)")
    ax2.set_title(f"Trial {trial_number} – Accuracy"); ax2.legend()

    plt.tight_layout()
    path = os.path.join(tempfile.gettempdir(), f"training_curves_trial_{trial_number}.png")
    fig.savefig(path, dpi=120)
    plt.close(fig)

    mlflow.log_artifact(path, artifact_path="plots")
    return path

In [120]:
# Obtain hyperparameters for this trial
def suggest_hyperparameters(trial):
    # Obtain the learning rate on a logarithmic scale
    lr = trial.suggest_float("lr", 1e-4, 1e-1, log=True)
    # Obtain the dropout ratio in a range from 0.0 to 0.9 with step size 0.1
    dropout = trial.suggest_float("dropout", 0.0, 0.9, step=0.1)
    # Obtain the optimizer to use by name
    optimizer_name = trial.suggest_categorical("optimizer_name", ["Adam", "Adadelta"])

    print(f"Suggested hyperparameters: \n{pformat(trial.params)}")
    return lr, dropout, optimizer_name

In [121]:
def get_mnist_dataloaders(batch_size=8):
    # Load the MNIST train and test datasets and save them to ./data
    mnist_train = datasets.MNIST('./data', train=True, download=True, transform=transforms.Compose([
                                               transforms.ToTensor(),
                                               transforms.Normalize((0.1307,), (0.3081,))
                                           ]))
    train_loader = torch.utils.data.DataLoader(mnist_train,
                                                       batch_size=batch_size,
                                                       shuffle=True)
    mnist_test = datasets.MNIST('./data', train=False, download=True, transform=transforms.Compose([
                                               transforms.ToTensor(),
                                               transforms.Normalize((0.1307,), (0.3081,))
                                           ]))
    val_loader = torch.utils.data.DataLoader(mnist_test,
                                                      batch_size=1000,
                                                      shuffle=True)
    return train_loader, val_loader

In [122]:
NUM_EPOCHS = 5

import time

def objective(trial):
    print("\n********************************\n")
    best_val_loss = float('Inf')
    best_val_acc  = 0.0                                                   
    
    # Start a new mlflow run
    with mlflow.start_run(run_name=f"trial-{trial.number}"):         
        mlflow.set_tags({
            "optuna_trial_number": trial.number,
            "framework": "pytorch",
            "task": "mnist_classification",
            "mlflow.note.content": f"Optuna trial {trial.number} for MNIST CNN"
        })

        # Get hyperparameter suggestions created by optuna and log them as params using mlflow
        lr, dropout, optimizer_name = suggest_hyperparameters(trial)
        mlflow.log_params(trial.params)
        mlflow.log_param("num_epochs", NUM_EPOCHS)                          

        # Use CUDA if GPU is available and log device as param using mlflow
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        mlflow.log_param("device", device)

        # Initialize network
        model = Net(dropout=dropout).to(device) # 

        model_summary = str(model)
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        summary_text = (
            f"Model architecture:\n{model_summary}\n\n"
            f"Total parameters    : {total_params:,}\n"
            f"Trainable parameters: {trainable_params:,}\n"
        )
        mlflow.log_text(summary_text, "model_summary.txt")
        mlflow.log_param("total_params", total_params)

        # -----------------------------------------------------------
        # Artifact: hyperparameters as JSON
        # -----------------------------------------------------------
        mlflow.log_text(json.dumps(trial.params, indent=2), "hyperparameters.json")

        # Pick an optimizer based on optuna's parameter suggestion
        if optimizer_name == "Adam":
            optimizer = optim.Adam(model.parameters(), lr=lr)
        if optimizer_name == "Adadelta":
            optimizer = optim.Adadelta(model.parameters(), lr=lr)
        scheduler = StepLR(optimizer, step_size=1, gamma=0.7)
        
        # Get DataLoaders for MNIST train and validation set
        train_loader, val_loader = get_mnist_dataloaders()
        
        # -----------------------------------------------------------
        # History dict for training-curve artifact
        # -----------------------------------------------------------
        history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": [], "forward_time": [], "backward_time": [], "val_forward_time": [], "timing_records": []}

        start_time = time.time() 

        # mlflow.log_text(f"Average forward and backward pass times will be logged for each epoch in model_timing.txt", "model_timing.txt")
        # Network training & validation loop
        for epoch in range(0, NUM_EPOCHS):
            avg_train_loss, train_acc, avg_train_forward_time, avg_train_backward_time = train(model, device, train_loader, optimizer, epoch)  
            avg_val_loss, val_acc, avg_val_forward_time = validate(model, device, val_loader)                   
            
            if avg_val_loss <= best_val_loss:
                best_val_loss = avg_val_loss
                best_val_acc  = val_acc                                   

            # Log average train and validation set loss metrics for the current epoch using mlflow
            mlflow.log_metric("avg_train_loss", avg_train_loss, step=epoch)
            mlflow.log_metric("avg_val_loss",   avg_val_loss,   step=epoch)

            mlflow.log_metric("train_accuracy", train_acc, step=epoch)
            mlflow.log_metric("val_accuracy",   val_acc,   step=epoch)
            mlflow.log_metric("forward_time_sec", avg_train_forward_time, step=epoch)
            mlflow.log_metric("backward_time_sec", avg_train_backward_time, step=epoch)
            mlflow.log_metric("val_forward_time_sec", avg_val_forward_time, step=epoch)


            # -----------------------------------------------------------
            trial.report(avg_val_loss, epoch)
            if trial.should_prune():
                mlflow.set_tag("optuna_pruned", True)
                mlflow.log_metric("best_val_loss", best_val_loss)
                mlflow.log_metric("best_val_accuracy", best_val_acc)
                raise optuna.TrialPruned()

            # Collect history for the curve plot
            history["train_loss"].append(avg_train_loss)
            history["val_loss"].append(avg_val_loss)
            history["train_acc"].append(train_acc)
            history["val_acc"].append(val_acc)
            history["forward_time"].append(avg_train_forward_time)
            history["backward_time"].append(avg_train_backward_time)
            history["val_forward_time"].append(avg_val_forward_time)
            history["timing_records"].append({
                "epoch": epoch,
                "forward_time_sec": avg_train_forward_time,
                "backward_time_sec": avg_train_backward_time,
                "val_forward_time_sec": avg_val_forward_time
            })
            scheduler.step()


        training_time = time.time() - start_time   
     
        mlflow.log_text(json.dumps(history["timing_records"], indent=2), "model_timing.json")

        # Final summary metrics
        mlflow.log_metric("best_val_loss",     best_val_loss)
        mlflow.log_metric("best_val_accuracy", best_val_acc)
        mlflow.log_metric("training_time_sec", training_time)
        mlflow.log_metric("avg_forward_time_sec", np.mean(history["forward_time"]))
        mlflow.log_metric("avg_backward_time_sec", np.mean(history["backward_time"]))
        mlflow.log_metric("avg_val_forward_time_sec", np.mean(history["val_forward_time"]))

        mlflow.pytorch.log_model(model, artifact_path="model")
        
        log_training_curves(history, trial.number)

        mlflow.set_tag("optuna_pruned", False)

    # Return the best validation loss achieved by the network.
    # This is needed as Optuna needs to know how the suggested hyperparameters are influencing the network loss.
    return best_val_loss

In [123]:
def main():
    torch.manual_seed(42)
    np.random.seed(42)
    
    pruner = optuna.pruners.MedianPruner(
        n_startup_trials=1,   # let at least 2 trials run fully before pruning
        n_warmup_steps=1,     # don't prune in the very first epoch
        interval_steps=1,
    )

    # Create the optuna study which shares the experiment name
    sampler = optuna.samplers.TPESampler(seed=42)
    study = optuna.create_study(
        study_name="pytorch-mlflow-optuna-2",
        direction="minimize",
        pruner=pruner,   
        sampler=sampler,
    )
    study.optimize(objective, n_trials=5)

    # Print optuna study statistics
    print("\n++++++++++++++++++++++++++++++++++\n")
    print("Study statistics: ")
    print("  Number of finished trials: ", len(study.trials))

    pruned   = [t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED]
    complete = [t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE]
    print(f"  Completed trials : {len(complete)}")
    print(f"  Pruned trials    : {len(pruned)}")

    print("\nBest trial:")
    trial = study.best_trial

    print("  Trial number: ", trial.number)
    print("  Loss (trial value): ", trial.value)

    print("  Params: ")
    for key, value in trial.params.items():
        print("    {}: {}".format(key, value))

    # ==================================================================
    with mlflow.start_run(run_name="optuna-study-summary"):

        mlflow.set_tags({
            "run_type": "study_summary",
            "best_trial_number": study.best_trial.number,
        })

        # Log best params as mlflow params
        mlflow.log_params({f"best_{k}": v for k, v in study.best_params.items()})
        mlflow.log_metric("best_trial_value", study.best_value)
        mlflow.log_metric("n_trials_complete", len(complete))
        mlflow.log_metric("n_trials_pruned",   len(pruned))

        # --- Study summary JSON artifact ---
        trials_summary = []
        for t in study.trials:
            trials_summary.append({
                "number": t.number,
                "value": t.value,
                "state": t.state.name,
                "params": t.params,
            })
        mlflow.log_text(json.dumps(trials_summary, indent=2), "study_trials_summary.json")

        # --- Optuna visualisation plots as artifacts ---
        tmpdir = tempfile.mkdtemp()

        try:
            fig = optuna.visualization.matplotlib.plot_optimization_history(study)
            fig.figure.savefig(os.path.join(tmpdir, "optimization_history.png"), dpi=120, bbox_inches="tight")
            plt.close(fig.figure)
        except Exception:
            pass

        try:
            fig = optuna.visualization.matplotlib.plot_param_importances(study)
            fig.figure.savefig(os.path.join(tmpdir, "param_importances.png"), dpi=120, bbox_inches="tight")
            plt.close(fig.figure)
        except Exception:
            pass

        try:
            fig = optuna.visualization.matplotlib.plot_parallel_coordinate(study)
            fig.figure.savefig(os.path.join(tmpdir, "parallel_coordinate.png"), dpi=120, bbox_inches="tight")
            plt.close(fig.figure)
        except Exception:
            pass

        # log the whole directory of plots at once
        mlflow.log_artifacts(tmpdir, artifact_path="optuna_plots")

        print("\n→ Study-level summary and plots logged to MLflow.")


In [ ]:
main()

[I 2026-02-20 00:08:15,975] A new study created in memory with name: pytorch-mlflow-optuna-2



********************************

Suggested hyperparameters: 
{'dropout': 0.9, 'lr': 0.0013292918943162175, 'optimizer_name': 'Adam'}
Train Epoch: 0 [0/60000 (0%)]	Loss: 2.237853


/Users/ayoolanjoku/opt/anaconda3/envs/dlasgn1/lib/python3.11/site-packages/torch/nn/modules/dropout.py:176: UserWarning: dropout2d: Received a 2-D input to dropout2d, which is deprecated and will result in an error in a future release. To retain the behavior and silence this warning, please use dropout instead. Note that dropout2d exists to provide channel-wise dropout on inputs with 2 spatial dimensions, a channel dimension, and an optional batch dimension (i.e. 3D or 4D inputs).
  return F.dropout2d(input, self.p, self.training, self.inplace)


Train Epoch: 0 [80/60000 (0%)]	Loss: 4.038299
Train Epoch: 0 [160/60000 (0%)]	Loss: 2.186963
Train Epoch: 0 [240/60000 (0%)]	Loss: 2.528416
Train Epoch: 0 [320/60000 (1%)]	Loss: 2.436603
Train Epoch: 0 [400/60000 (1%)]	Loss: 2.037756
Train Epoch: 0 [480/60000 (1%)]	Loss: 2.292862
Train Epoch: 0 [560/60000 (1%)]	Loss: 2.116280
Train Epoch: 0 [640/60000 (1%)]	Loss: 2.415453
Train Epoch: 0 [720/60000 (1%)]	Loss: 2.303910
Train Epoch: 0 [800/60000 (1%)]	Loss: 2.187627
Train Epoch: 0 [880/60000 (1%)]	Loss: 2.297383
Train Epoch: 0 [960/60000 (2%)]	Loss: 2.337281
Train Epoch: 0 [1040/60000 (2%)]	Loss: 2.304430
Train Epoch: 0 [1120/60000 (2%)]	Loss: 2.298063
Train Epoch: 0 [1200/60000 (2%)]	Loss: 2.298986
Train Epoch: 0 [1280/60000 (2%)]	Loss: 2.188573
Train Epoch: 0 [1360/60000 (2%)]	Loss: 2.407151
Train Epoch: 0 [1440/60000 (2%)]	Loss: 2.180109
Train Epoch: 0 [1520/60000 (3%)]	Loss: 2.407701
Train Epoch: 0 [1600/60000 (3%)]	Loss: 2.503025
Train Epoch: 0 [1680/60000 (3%)]	Loss: 2.291662
Train

2026/02/20 00:11:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 00:11:40 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


...

Test set: Average loss: 0.5503, Accuracy: 8672/10000 (87%)

🏃 View run trial-1 at: http://127.0.0.1:5002/#/experiments/3/runs/627ef629f5654d17b417a6a2d3824796
🧪 View experiment at: http://127.0.0.1:5002/#/experiments/3

********************************

Suggested hyperparameters: 
{'dropout': 0.7000000000000001,
 'lr': 0.006358358856676255,
 'optimizer_name': 'Adadelta'}
Train Epoch: 0 [0/60000 (0%)]	Loss: 2.311469
Train Epoch: 0 [80/60000 (0%)]	Loss: 2.299970
Train Epoch: 0 [160/60000 (0%)]	Loss: 2.357681
Train Epoch: 0 [240/60000 (0%)]	Loss: 2.392249
Train Epoch: 0 [320/60000 (1%)]	Loss: 2.187794
Train Epoch: 0 [400/60000 (1%)]	Loss: 2.339608
Train Epoch: 0 [480/60000 (1%)]	Loss: 2.311393
Train Epoch: 0 [560/60000 (1%)]	Loss: 2.460608
Train Epoch: 0 [640/60000 (1%)]	Loss: 2.419699
Train Epoch: 0 [720/60000 (1%)]	Loss: 2.390084
Train Epoch: 0 [800/60000 (1%)]	Loss: 2.179348
Train Epoch: 0 [880/60000 (1%)]	Loss: 2.241987
Train Epoch: 0 [960/60000 (2%)]	Loss: 2.413857
Train Epoch: 0 [10

2026/02/20 00:16:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/02/20 00:16:55 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.


Test set: Average loss: 0.2965, Accuracy: 9117/10000 (91%)



[I 2026-02-20 00:16:57,138] Trial 2 finished with value: 0.2964892227172852 and parameters: {'lr': 0.006358358856676255, 'dropout': 0.7000000000000001, 'optimizer_name': 'Adadelta'}. Best is trial 2 with value: 0.2964892227172852.


🏃 View run trial-2 at: http://127.0.0.1:5002/#/experiments/3/runs/c4be8722971f491982782b00294fbf64
🧪 View experiment at: http://127.0.0.1:5002/#/experiments/3

********************************

Suggested hyperparameters: 
{'dropout': 0.2, 'lr': 0.03142880890840111, 'optimizer_name': 'Adadelta'}
Train Epoch: 0 [0/60000 (0%)]	Loss: 2.320484
Train Epoch: 0 [80/60000 (0%)]	Loss: 2.212327
Train Epoch: 0 [160/60000 (0%)]	Loss: 2.260451
Train Epoch: 0 [240/60000 (0%)]	Loss: 2.415076
Train Epoch: 0 [320/60000 (1%)]	Loss: 1.926691
Train Epoch: 0 [400/60000 (1%)]	Loss: 1.652249
Train Epoch: 0 [480/60000 (1%)]	Loss: 1.568988
Train Epoch: 0 [560/60000 (1%)]	Loss: 1.426912
Train Epoch: 0 [640/60000 (1%)]	Loss: 1.147028
Train Epoch: 0 [720/60000 (1%)]	Loss: 1.222860
Train Epoch: 0 [800/60000 (1%)]	Loss: 0.949941
Train Epoch: 0 [880/60000 (1%)]	Loss: 0.419499
Train Epoch: 0 [960/60000 (2%)]	Loss: 1.107059
Train Epoch: 0 [1040/60000 (2%)]	Loss: 0.693337
Train Epoch: 0 [1120/60000 (2%)]	Loss: 0.551032


In [ ]:
# Search for the run with the lowest best_val_loss across all trial runs
experiment = mlflow.get_experiment_by_name(EXPERIMENT_NAME)
best_run = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    filter_string="tags.run_type != 'study_summary'",
    order_by=["metrics.best_val_loss ASC"],
    max_results=1,
)

if not best_run.empty:
    best_run_id = best_run.iloc[0]["run_id"]
    print(f"Best run ID          : {best_run_id}")
    print(f"Best validation loss : {best_run.iloc[0]['metrics.best_val_loss']:.4f}")
    print(f"Best validation acc  : {best_run.iloc[0]['metrics.best_val_accuracy']:.2f}%")
    
    # Load the model back
    loaded_model = mlflow.pytorch.load_model(f"runs:/{best_run_id}/model")
    loaded_model.eval()
    print("\n✓ Model loaded successfully from MLflow — ready for inference.")
else:
    print("No completed runs found.")

No completed runs found.
